## Kerendia NBRX Calculation from Raw IQVIA LAAD FCT_RX

**Purpose:** Compute HCP-level Kerendia NBRX and TRX from raw Snowflake claims (`STG_CAR_IQV_LAAD_FCT_RX`)  
**Scope:** Kerendia only, all claim types (PD + RV + RJ), **full history** (no date window)  
**NBRX Logic:** First-ever claim per patient-product = NBRX; or gap > 365 days from last claim = NBRX  
**Output:** HCP-level table with paid NBRX, overall NBRX, and total TRX  
**Reference:** `Main_NBRX_Calculation_Pipeline` (same NBRX logic, but for all products with lookback window)

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col, min as spark_min, lag, expr, when, row_number, sum as spark_sum
from datetime import datetime, timedelta

sql("SET spark.databricks.delta.formatCheck.enabled=false")

In [0]:
# ============================================================
# Query raw FCT_RX for KERENDIA only
# - Full history (no date window)
# - All claim types (PD, RV, RJ)
# - Joins with DIM_PRD to filter PRODUCT_GROUP = 'KERENDIA'
# ============================================================
query_kerendia = '''
    SELECT DISTINCT
        A.CLAIM_ID,
        A.PATIENT_ID,
        TO_DATE(A.SVC_DT) AS SVC_DT,
        A.PROVIDER_ID,
        A.CLAIM_TYPE,
        TRIM(UPPER(B.PRODUCT_GROUP)) AS PRODUCT_GROUP
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE TRIM(UPPER(B.PRODUCT_GROUP)) = 'KERENDIA'
'''

kerendia_claims = Get_Data_Snowflakes(query_kerendia)
kerendia_claims = kerendia_claims.withColumn("PROVIDER_ID", col("PROVIDER_ID").cast("string"))
print("Kerendia claims (full history, all types):")
print(f"  Shape: {kerendia_claims.count():,} rows, {len(kerendia_claims.columns)} cols")
print(f"  Date range: {kerendia_claims.agg(F.min('SVC_DT')).collect()[0][0]} to {kerendia_claims.agg(F.max('SVC_DT')).collect()[0][0]}")
print(f"  Distinct HCPs: {kerendia_claims.select('PROVIDER_ID').distinct().count():,}")
print(f"  Distinct patients: {kerendia_claims.select('PATIENT_ID').distinct().count():,}")

# Claim type breakdown
print("\nClaim type breakdown:")
kerendia_claims.groupBy("CLAIM_TYPE").count().orderBy(F.desc("count")).show()

In [0]:
# ============================================================
# NBRX Calculation Logic (from Main_NBRX_Calculation_Pipeline)
# - First-ever claim per patient-product = NBRX
# - Gap > 365 days from last claim = NBRX
# - Only first claim per patient-product-date counts
# ============================================================

# Step 1: Dedup - pick first claim per patient-product-date
dedup_window = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP", "SVC_DT").orderBy("CLAIM_ID")
kerendia_claims = kerendia_claims.withColumn("CLAIM_RANK", row_number().over(dedup_window))

# Step 2: Window for NBRX - per patient-product history
window_spec = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP").orderBy("SVC_DT")

# First-ever claim date per patient-product
kerendia_claims = kerendia_claims.withColumn("MIN_SVC_DT", spark_min("SVC_DT").over(window_spec))

# Previous claim date
kerendia_claims = kerendia_claims.withColumn("LAG_SVC_DT", lag("SVC_DT").over(window_spec))

# Step 3: NBRX_FLAG - first claim per day only
kerendia_claims_w_nbrx = kerendia_claims.withColumn(
    "NBRX_FLAG",
    when((col("MIN_SVC_DT") == col("SVC_DT")) & (col("CLAIM_RANK") == 1), 1)
    .when((col("LAG_SVC_DT").isNotNull()) & (col("LAG_SVC_DT") < expr("date_add(SVC_DT, -365)")) & (col("CLAIM_RANK") == 1), 1)
    .otherwise(0)
)

# Step 4: Cleanup
kerendia_claims_w_nbrx = kerendia_claims_w_nbrx.drop("MIN_SVC_DT", "LAG_SVC_DT", "CLAIM_RANK")
print(f"Claims with NBRX flag: {kerendia_claims_w_nbrx.count():,}")
print(f"Total NBRX = 1 rows: {kerendia_claims_w_nbrx.filter(col('NBRX_FLAG') == 1).count():,}")

In [0]:
# ============================================================
# Aggregate to HCP level:
# - NBRX_KERENDIA_OVERALL:  NBRX across all claim types
# - NBRX_KERENDIA_PD:       NBRX from paid claims only
# - TRX_KERENDIA_OVERALL:   Total claims (= TRX) all types
# - TRX_KERENDIA_PD:        Total claims paid only
# ============================================================
hcp_kerendia_rollup = (
    kerendia_claims_w_nbrx
    .groupBy("PROVIDER_ID")
    .agg(
        # NBRX - all claim types
        spark_sum(
            when(col("NBRX_FLAG") == 1, 1).otherwise(0)
        ).alias("NBRX_KERENDIA_OVERALL"),
        # NBRX - paid only
        spark_sum(
            when((col("NBRX_FLAG") == 1) & (col("CLAIM_TYPE") == "PD"), 1).otherwise(0)
        ).alias("NBRX_KERENDIA_PD"),
        # TRX - all claim types (each claim = 1 TRx)
        F.count("*").alias("TRX_KERENDIA_OVERALL"),
        # TRX - paid only
        spark_sum(
            when(col("CLAIM_TYPE") == "PD", 1).otherwise(0)
        ).alias("TRX_KERENDIA_PD"),
        # Date range per HCP
        F.min("SVC_DT").alias("FIRST_KERENDIA_CLAIM"),
        F.max("SVC_DT").alias("LAST_KERENDIA_CLAIM")
    )
)

hcp_ct = hcp_kerendia_rollup.count()
print(f"HCP-level Kerendia rollup: {hcp_ct:,} HCPs")
print(f"\nWriter counts (NBRX > 0):")
print(f"  Overall NBRX > 0: {hcp_kerendia_rollup.filter(col('NBRX_KERENDIA_OVERALL') > 0).count():,}")
print(f"  Paid NBRX > 0:    {hcp_kerendia_rollup.filter(col('NBRX_KERENDIA_PD') > 0).count():,}")
print(f"\nWriter counts (TRX > 0):")
print(f"  Overall TRX > 0:  {hcp_kerendia_rollup.filter(col('TRX_KERENDIA_OVERALL') > 0).count():,}")
print(f"  Paid TRX > 0:     {hcp_kerendia_rollup.filter(col('TRX_KERENDIA_PD') > 0).count():,}")

In [0]:
# ============================================================
# Summary stats
# ============================================================
print("HCP-Level Summary:")
hcp_kerendia_rollup.select(
    F.count("*").alias("total_hcps"),
    spark_sum("NBRX_KERENDIA_OVERALL").alias("total_nbrx_overall"),
    spark_sum("NBRX_KERENDIA_PD").alias("total_nbrx_pd"),
    spark_sum("TRX_KERENDIA_OVERALL").alias("total_trx_overall"),
    spark_sum("TRX_KERENDIA_PD").alias("total_trx_pd"),
    F.min("FIRST_KERENDIA_CLAIM").alias("earliest_claim"),
    F.max("LAST_KERENDIA_CLAIM").alias("latest_claim")
).display()

# Claim type breakdown
print("\nClaim type breakdown:")
hcp_kerendia_rollup.select(
    F.count(when(col("NBRX_KERENDIA_PD") > 0, 1)).alias("HCPs_w_paid_NBRX"),
    F.count(when((col("NBRX_KERENDIA_OVERALL") > 0) & (col("NBRX_KERENDIA_PD") == 0), 1)).alias("HCPs_w_nonpaid_NBRX_only"),
    F.count(when((col("TRX_KERENDIA_PD") > 0) & (col("NBRX_KERENDIA_PD") == 0), 1)).alias("HCPs_w_paid_TRX_but_no_paid_NBRX"),
).display()

In [0]:
# ============================================================
# Save to Volume for use in V6 HCP Universe notebook
# ============================================================
OUTPUT_PATH = (
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
    "dev_folder_3_1/notebook_output/"
    "kerendia_nbrx_from_raw_fct_rx"
)

hcp_kerendia_rollup.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(OUTPUT_PATH)
print(f"Saved to: {OUTPUT_PATH}")
print(f"Rows: {spark.read.format('delta').load(OUTPUT_PATH).count():,}")